# BÁO CÁO THỰC TẬP CƠ SỞ - KẾ HOẠCH 4
So sánh hiệu năng CNN, RNN, LSTM trên bộ dữ liệu IMDB Sentiment Analysis (Kaggle)

In [ ]:
import os

# Giới hạn TensorFlow chỉ sử dụng GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Bật memory growth để TF chỉ cấp phát bộ nhớ khi cần
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Đã cấu hình TensorFlow sử dụng GPU 0 với memory growth.")
    except RuntimeError as e:
        print(e)

## 1. Chương 1: Tiền xử lý dữ liệu
Tải file CSV từ Kaggle, làm sạch văn bản (loại bỏ HTML, dấu câu, số...), mã hóa và padding để chuẩn bị đưa vào mô hình.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

MAX_WORDS = 10000  # Giới hạn số lượng từ vựng
MAX_LEN = 200      # Giới hạn chiều dài mỗi đoạn text

# 1. Load data
print('Đọc dữ liệu từ file CSV...')
df = pd.read_csv('dataset/IMDB Dataset.csv')
print(f'Số lượng mẫu ban đầu: {len(df)}')
print(df.head())

# 2. Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text) # Bỏ nội dung trong ngoặc vuông
    text = re.sub('https?://\S+|www\.\S+', '', text) # Bỏ URL
    text = re.sub('<.*?>+', '', text) # Bỏ thẻ HTML
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # Bỏ dấu câu
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text) # Bỏ chữ chứa số
    return text

print('\nĐang làm sạch văn bản...')
df['review'] = df['review'].apply(clean_text)

# 3. Encode Labels (positive -> 1, negative -> 0)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
texts = df['review'].values
labels = df['sentiment'].values

# 4. Tokenization (Chuyển text thành số)
print('Đang Tokenize văn bản...')
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# 5. Padding (Cắt / Thêm số 0 để các mảng dài bằng nhau)
data = pad_sequences(sequences, maxlen=MAX_LEN)

# 6. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

print('\nHoàn tất tiền xử lý!')
print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

I0000 00:00:1777538124.731983 3685773 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777538124.768569 3685773 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777538125.653286 3685773 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Đọc dữ liệu từ file CSV...
Số lượng mẫu ban đầu: 50000
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Đang làm sạch văn bản...


## 2. Chương 2: Xây dựng và Huấn luyện CNN cho Text

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

def build_cnn():
    model = Sequential([
        Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()


In [ ]:
# Huấn luyện CNN
print('Bắt đầu huấn luyện CNN...')
cnn_history = cnn_model.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.2)

## 3. Chương 3: Xây dựng và Huấn luyện RNN

In [ ]:
from tensorflow.keras.layers import SimpleRNN

def build_rnn():
    model = Sequential([
        Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
        SimpleRNN(64),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

rnn_model = build_rnn()
rnn_model.summary()


In [ ]:
# Huấn luyện RNN
print('Bắt đầu huấn luyện RNN...')
rnn_history = rnn_model.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.2)

## 4. Chương 4: Xây dựng và Huấn luyện LSTM

In [ ]:
from tensorflow.keras.layers import LSTM

def build_lstm():
    model = Sequential([
        Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
        LSTM(64),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_lstm()
lstm_model.summary()


In [ ]:
# Huấn luyện LSTM
print('Bắt đầu huấn luyện LSTM...')
lstm_history = lstm_model.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.2)

## 5. Phần Kết: Đánh giá và So sánh

In [ ]:
print('Đánh giá CNN...')
cnn_loss, cnn_acc = cnn_model.evaluate(X_test, y_test, verbose=0)

print('Đánh giá RNN...')
rnn_loss, rnn_acc = rnn_model.evaluate(X_test, y_test, verbose=0)

print('Đánh giá LSTM...')
lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)

models = ['CNN', 'RNN', 'LSTM']
accuracies = [cnn_acc, rnn_acc, lstm_acc]

plt.figure(figsize=(8, 5))
plt.bar(models, accuracies, color=['blue', 'orange', 'green'])
plt.title('So sánh độ chính xác của CNN, RNN, LSTM trên tập Test')
plt.ylabel('Độ chính xác')
plt.ylim([0, 1.0])
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.02, str(round(v, 4)), ha='center', fontweight='bold')

# Lưu biểu đồ để đưa vào báo cáo LaTeX
plt.savefig('comparison_chart.png')
plt.show()
